# Seven-Arm Dependency-Aware Selective Regeneration Benchmark

**SMOKE / NON-PUBLICATION**

Runs the benchmark entirely from attached Kaggle Datasets.  No GitHub clone.

- **Smoke profile** (default): 1 scenario, 7 strategies, non-publication.
- **Pilot profile** (requires `--profile pilot`): 12 scenarios, 2 strategies, 2 reps.
- **Research profile** (requires `--profile research`): 24 scenarios, 4 strategies, 3 reps.

Only smoke runs automatically.  Pilot and research require manual `--profile` selection.

In [ ]:
import sys
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/runs")

# ----- known candidate paths ---
KNOWN_CODE = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-code"
KNOWN_DATA = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-data"
KNOWN_MODEL = KAGGLE_INPUT / "models/qwen-lm/qwen2.5-coder/transformers/7b-instruct/1"

# simplified fallback paths (Kaggle Dataset slug style)
FALLBACK_CODE = KAGGLE_INPUT / "dependency-aware-selective-regeneration-code"
FALLBACK_DATA = KAGGLE_INPUT / "dependency-aware-selective-regeneration-data"

def discover(label, candidates, required_subdir=None):
    for p in candidates:
        if p.is_dir():
            if required_subdir is None or (p / required_subdir).is_dir():
                return p
            print(f"  [info] {p.name} exists but missing '{required_subdir}'")
    # fallback: scan top-level entries
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if entry.is_dir() and (required_subdir is None or (entry / required_subdir).is_dir()):
                return entry
    raise FileNotFoundError(f"Cannot find {label} in {KAGGLE_INPUT}")

# 1a. Code dataset ---
CODE_DIR = discover("code dataset", [KNOWN_CODE, FALLBACK_CODE], required_subdir="src")
print(f"Code dataset: {CODE_DIR}")

src_dir = CODE_DIR / "src"
if src_dir.is_dir():
    sys.path.insert(0, str(src_dir))
    print(f"Added to sys.path: {src_dir}")
else:
    raise FileNotFoundError(f"src/ not found in code dataset: {CODE_DIR}")

# 1b. Data dataset ---
DATA_DIR = discover("data dataset", [KNOWN_DATA, FALLBACK_DATA], required_subdir="scenarios")
print(f"Data dataset: {DATA_DIR}")

# 1c. Model path ---
if KNOWN_MODEL.is_dir():
    MODEL_PATH = str(KNOWN_MODEL.resolve())
    print(f"Qwen model: {MODEL_PATH}")
else:
    MODEL_PATH = ""
    print("WARNING: Qwen model not found at known path")

# 1d. Find requirements-kaggle.txt in code dataset ---
req_txt = CODE_DIR / "requirements-kaggle.txt"
if req_txt.is_file():
    print(f"Installing from {req_txt} ...")
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_txt)],
        check=True,
    )
    print("Dependencies installed.")
else:
    print(f"No requirements-kaggle.txt found in {CODE_DIR} -- skipping pip install")

print("\nSetup complete.")

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", None))
    if gpu_mem:
        print(f"GPU memory: {gpu_mem / 1024**3:.1f} GB")
    else:
        print("GPU memory: <unknown>")
else:
    print("WARNING: No GPU available. Benchmark will run on CPU.")

from pathlib import Path
model_dir = Path(MODEL_PATH) if MODEL_PATH else None
if model_dir and model_dir.is_dir():
    files = [f.name for f in model_dir.iterdir() if f.is_file()]
    has_config = "config.json" in files
    has_weights = any(f.endswith((".safetensors", ".bin", ".pt")) for f in files)
    print(f"Model path: {model_dir}")
    print(f"  config.json: {'OK' if has_config else 'MISSING'}")
    print(f"  weights: {'OK' if has_weights else 'checking deeper'}")
    if not has_weights:
        for child in model_dir.iterdir():
            if child.is_dir():
                child_files = [f.name for f in child.iterdir() if f.is_file()]
                if any(f.endswith((".safetensors", ".bin", ".pt")) for f in child_files):
                    print(f"    found in subdirectory: {child.name}")
                    has_weights = True
    if not (has_config and has_weights):
        print("WARNING: Model directory may be incomplete.")
else:
    print("WARNING: Qwen model not found. Real execution will fail.")
    print("Dry-run will still work.")

In [ ]:
import os
from huggingface_hub import HfApi

# Read HF_TOKEN from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# Export environment variables
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_RESULTS_REPO_ID"] = "NabilDo/selective-regeneration-experiment-results"

print("HF_TOKEN: [exported to environment]")
print(f"HF_RESULTS_REPO_ID: {os.environ['HF_RESULTS_REPO_ID']}")

# Authenticate with Hugging Face
api = HfApi(token=HF_TOKEN)
who = api.whoami()
print(f"Authenticated as: {who['name']}")

# Verify the Dataset repository is private
repo_id = os.environ["HF_RESULTS_REPO_ID"]
info = api.repo_info(repo_id, repo_type="dataset")
if info.private:
    print(f"Repository '{repo_id}' is PRIVATE -- OK")
else:
    print(f"WARNING: Repository '{repo_id}' is NOT private -- results will be public")

print("\nHugging Face setup complete.")

---
### Session 1: Smoke validation with Hugging Face sync

Runs smoke profile (1 scenario, 7 strategies, 1 rep) with `--hf-sync` enabled.
Results are uploaded to Hugging Face after every completed run.

Qwen model loads with `local_files_only=True` (no HF model download).

In [ ]:
import subprocess, sys, os

script = CODE_DIR / "seven_arm_benchmark.py"
if not script.is_file():
    raise FileNotFoundError(f"seven_arm_benchmark.py not found in {CODE_DIR}")

env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_DIR / "src") + (
    os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
)

SOURCE_TAG = "v0.7.0-smoke-passed"
session1_cmd = [
    sys.executable, str(script),
    "--profile", "smoke",
    "--max-runs", "1",
    "--hf-sync",
    "--source-tag", SOURCE_TAG,
    "--hf-repo-id", "NabilDo/selective-regeneration-experiment-results",
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
print("Session 1 command:", " ".join(session1_cmd))
print("\n--- Session 1 output ---")
result = subprocess.run(session1_cmd, capture_output=True, text=True, env=env)
print(result.stdout)
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError("Session 1 failed")

In [ ]:
import json
from pathlib import Path

run_dir = OUTPUT_DIR
print(f"Output directory: {run_dir}")
print()

# Experiment ID
exp_id_path = run_dir / "experiment_id.txt"
if exp_id_path.exists():
    print(f"Experiment ID: {exp_id_path.read_text().strip()}")
else:
    print("Experiment ID: (not saved yet)")
print()

# Checkpoint files
for name in ["checkpoint.json", "progress.json", "remote_sync.json"]:
    p = run_dir / name
    if p.exists():
        data = json.loads(p.read_text())
        print(f"{name}: {json.dumps(data, indent=2)[:500]}")
    else:
        print(f"{name}: (not found)")
print()

# run_records.jsonl
records_path = run_dir / "run_records.jsonl"
if records_path.exists():
    lines = records_path.read_text().strip().split("\n")
    print(f"run_records.jsonl: {len(lines)} record(s)")
    for l in lines:
        rec = json.loads(l)
        rid = rec.get("run_id", "?")
        status = rec.get("status", "?")
        print(f"  {rid}: {status}")
else:
    print("run_records.jsonl: (not found)")
print()

# Count completed vs pending
completed = [l for l in (records_path.read_text().strip().split("\n") if records_path.exists() else []) if l]
completed_ids = []
for l in completed:
    rec = json.loads(l)
    if rec.get("status") == "completed":
        completed_ids.append(rec.get("run_id", "?"))
pending_count = 7 - len(completed)
print(f"Completed runs: {len(completed_ids)}")
for rid in completed_ids:
    print(f"  {rid}")
print(f"Pending runs: {max(0, pending_count)}")

---
### Session 2: Resume from Hugging Face

Resumes the experiment using `--resume-from-hf` and the experiment ID from Session 1.
The benchmark will download the previous recovery state from Hugging Face and continue.

Expected: the first run (already completed in Session 1) is skipped, and one new run executes.

In [ ]:
import subprocess, sys, os

# Read experiment ID from Session 1 output
exp_id_path = OUTPUT_DIR / "experiment_id.txt"
if exp_id_path.exists():
    EXPERIMENT_ID = exp_id_path.read_text().strip()
    print(f"Experiment ID: {EXPERIMENT_ID}")
else:
    EXPERIMENT_ID = input("Enter experiment ID from Session 1: ").strip()

session2_cmd = [
    sys.executable, str(script),
    "--profile", "smoke",
    "--resume-from-hf",
    "--experiment-id", EXPERIMENT_ID,
    "--max-runs", "1",
    "--hf-sync",
    "--source-tag", SOURCE_TAG,
    "--hf-repo-id", "NabilDo/selective-regeneration-experiment-results",
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
print("Session 2 command:", " ".join(session2_cmd))
print("\n--- Session 2 output ---")
result = subprocess.run(session2_cmd, capture_output=True, text=True, env=env)
print(result.stdout)
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError("Session 2 failed")

In [ ]:
import json
from pathlib import Path

records_path = OUTPUT_DIR / "run_records.jsonl"
if not records_path.exists():
    raise FileNotFoundError("run_records.jsonl not found")

lines = records_path.read_text().strip().split("\n")
all_records = [json.loads(l) for l in lines if l]
succeeded_records = [r for r in all_records if r.get("status") == "succeeded"]

print(f"Total records: {len(all_records)}")
print(f"Succeeded: {len(succeeded_records)}")

# Check succeeded count is 2
if len(succeeded_records) == 2:
    print("PASS: succeeded count is 2")
else:
    print("FAIL: expected 2 succeeded runs")

# Check no duplicated run IDs
all_ids = [r.get("run_id", "") for r in succeeded_records]
if len(all_ids) == len(set(all_ids)):
    print("PASS: no duplicated run IDs")
else:
    dupes = [rid for rid in all_ids if all_ids.count(rid) > 1]
    print(f"FAIL: duplicate run IDs found: {set(dupes)}")

# Check first run is skipped during resume
for r in succeeded_records:
    print(f"  {r.get('run_id', '?')}: {r.get('status', '?')}")

if len(succeeded_records) >= 2:
    first_rid = succeeded_records[0].get("run_id", "")
    second_rid = succeeded_records[1].get("run_id", "")
    if first_rid != second_rid:
        print(f"PASS: run IDs are distinct ({first_rid} != {second_rid})")
    else:
        print(f"FAIL: duplicate run ID detected: {first_rid}")

# Hugging Face remote commit
sync_path = OUTPUT_DIR / "remote_sync.json"
if sync_path.exists():
    sync_data = json.loads(sync_path.read_text())
    print(f"HF remote commit: {sync_data.get('last_commit_url', 'unknown')}")
    print(f"HF remote path prefix: {sync_data.get('remote_path', 'unknown')}")

print("\nValidation complete.")

## Notes

- **Smoke**: default profile (1 scenario x 7 strategies, non-publication).
- **Pilot**: `--profile pilot` -- 12 scenarios, agent + selective, 2 reps.
- **Research**: `--profile research` -- 24 scenarios, 4 strategies, 3 reps.
- All outputs go to `/kaggle/working/runs/`.
- Internet is required only for Hugging Face result synchronization.
- HF_TOKEN is required only for `--hf-sync` and `--resume-from-hf`.
- Qwen model loading remains offline from the attached Kaggle Model.

## Disabled profiles

Pilot and research remain disabled for automated runs.

To enable pilot:

    Edit Session 1 cell: change `--profile smoke` to `--profile pilot`

To enable research:

    Edit Session 1 cell: change `--profile smoke` to `--profile research`

Both require removing `--max-runs 1` for a full profile execution.